
# DS-A4: Model Comparison and Calibration Lab

## 1. Problem Statement

This study aims to evaluate and compare machine-learning classification models using in-situ geochemical data from phosphatic components of the Early Cambrian Abbottabad Formation, Kakul-Garlaniyan, Pakistan.

The study uses EPMA major-element compositions and LA-ICP-MS trace-element geochemical data to investigate whether different phosphatic components can be distinguished using their geochemical characteristics.

## 2. Research Question

Can machine-learning models distinguish oolite cores, oolite rims, pellet cores, and pellet rims using their in-situ geochemical compositions, and how reliable are their predicted probabilities?

## 3. Study Objective

The objective is to compare simple and complex classification models under an identical evaluation protocol, assess their predictive and probability-calibration performance, analyze classification errors, and determine an appropriate classification threshold based on error costs and human review capacity.

## 4. Dataset

The analysis will use Supplementary File 2 from the published Journal of Asian Earth Sciences study. The dataset contains in-situ EPMA and LA-ICP-MS measurements of phosphatic components.

The mineral phase will be investigated as the candidate target variable.


In [1]:

# 1. Import required libraries

import pandas as pd
import numpy as np

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Libraries imported successfully.
Pandas version: 3.1.1
NumPy version: 2.5.4


In [2]:

# 2. Define the dataset location

file_path = Path(
    r"E:\DS_4\data\Supplementray_Files.xlsx"
)

# Check whether the file exists

if file_path.exists():
    print("Dataset found successfully!")
    print("File:", file_path.name)
    print("File size:", round(file_path.stat().st_size / 1024, 2), "KB")
else:
    raise FileNotFoundError(
        "Supplementary Excel file was not found. Check the file path."
    )


Dataset found successfully!
File: Supplementray_Files.xlsx
File size: 128.73 KB


In [3]:

# 3. Load Supplementary File 2 only

raw_data = pd.read_excel(
    file_path,
    sheet_name="Supplementary file 2",
    header=2
)

print("Supplementary File 2 loaded successfully!")

print("Number of observations:", raw_data.shape[0])
print("Number of columns:", raw_data.shape[1])


Supplementary File 2 loaded successfully!
Number of observations: 103
Number of columns: 75


In [4]:

# 4. Display the first 10 observations

display(raw_data.head(10))


,Unnamed: 0,Points,Mineral phase,F,Na2O,MgO,FeO,P2O5,CaO,MnO,Al2O3,SiO2,Total,Li,Be,B,Sc,V,Ca,Cr,Co,Ni,Cu,Zn,Ga,Ge,As,Se,Rb,Sr,Zr,Nb,Mo,Ag,Cd,In,Sn,Sb,Cs,Ba,Hf,Ta,W,Au,Tl,Bi,Pb,Th,U,La,Ce,Pr,Nd,Sm,Eu,Gd,Tb,Dy,Y,Ho,Er,Tm,Yb,Lu,∑REY,Ce/Ce*,Pr/Pr*,Y/Y*,Eu/Eu*,La/Yb,La/Sm,Y/Ho,87Sr/86Sr Ini,Sr/Ca,Rb/Sr
0,LPL,2-1,Oolite core,4.039,0.287,0.042,0.010,39.977,54.316,0.007,0.136003,0.496413,97.474,4.495539,1.781211,6.050487,0.751170,10.257750,388192.810271,11.139925,0.196778,0.540819,1.234765,7.993858,1.617245,0.781115,3.265495,1.121116,1.059296,1154.983480,6.577110,0.064008,0.187814,0.000000,0.079154,0.000000,0.000397,0.107858,0.076229,199.543504,0.153778,0.005533,0.085979,0.008927,0.027155,0.016466,2.451133,1.005189,28.247229,51.260437,50.891498,13.226631,65.334556,12.735382,3.089331,18.718714,2.560017,18.778042,278.915941,4.588878,13.243609,1.564047,8.912145,1.128986,544.948213,0.391133,1.167234,2.390437,0.906430,0.424606,0.584790,60.780860,0.711625,0.002975,0.000917
1,NaN,2-2,NaN,3.725,0.262,0.028,0.025,40.828,54.754,0.013,0.034185,0.798524,98.560,1.987853,1.100990,3.391203,0.446459,3.631258,391323.166904,1.708331,0.028021,0.402554,0.774975,5.120551,1.350201,0.531312,4.120102,1.738702,0.211350,984.662923,3.132744,0.051540,0.225173,0.019901,0.000000,0.003430,0.126899,0.039542,0.039669,113.526066,0.050290,0.003876,0.106593,0.003164,0.016977,0.013463,2.427896,0.837472,23.359534,53.404014,51.373959,12.992344,61.719437,12.404207,3.293055,19.110671,2.572587,19.499122,258.000488,4.255500,13.377660,1.606674,8.313308,0.978360,522.901385,0.401039,1.193322,2.258834,0.962477,0.474226,0.625510,60.627542,NaN,NaN,NaN
2,NaN,2-3,NaN,3.959,0.306,0.066,0.015,40.352,54.492,0.005,0.031026,0.148559,98.047,3.272041,2.525362,4.954941,0.593160,7.293574,389450.670471,5.532900,0.071200,0.113809,0.790126,8.669151,1.617404,0.973738,3.189984,2.816809,0.265243,894.544240,2.527632,0.018104,0.301992,0.015715,0.000000,0.011240,0.042666,0.120505,0.000000,173.748182,0.054682,0.001197,0.102397,0.000000,0.000000,0.006136,1.753225,1.379267,29.895780,69.144898,65.882511,16.661532,74.982110,15.240629,3.829297,22.934031,3.114236,24.416176,330.008920,5.656887,17.536755,1.998253,9.973477,1.381669,662.761382,0.411578,1.241588,2.237458,0.924847,0.511799,0.659154,58.337554,NaN,NaN,NaN
3,NaN,5-1,NaN,3.976,0.211,0.096,0.021,41.057,54.716,0.000,0.105539,1.471582,98.957,3.251387,1.491991,7.558724,0.578535,9.712137,391051.583452,15.054818,0.071104,0.169200,1.231533,4.401766,1.453536,1.058669,3.518025,1.528631,0.943659,1091.214559,6.594636,0.090181,0.299828,0.022757,0.311476,0.013492,0.077779,0.030523,0.018909,283.991039,0.096025,0.001278,0.086544,0.000000,0.002632,0.024258,1.840303,1.231757,25.415525,57.419136,53.659086,13.621282,63.129541,14.064289,3.190684,19.791429,2.703266,20.189824,269.548655,4.701868,13.666707,1.622296,8.898514,1.201576,547.408153,0.400618,1.216411,2.204148,0.871332,0.476349,0.593155,57.327998,0.711681,0.002790,0.000865
4,NaN,5-2,NaN,4.078,0.306,0.063,0.029,40.236,54.642,0.012,0.129656,0.384135,98.194,4.342306,3.166016,6.292931,0.360237,14.899902,390522.710414,14.199617,0.000000,0.118146,1.128739,8.336341,1.211591,0.601210,2.663321,1.835701,1.011092,996.299749,3.318590,0.083208,0.600033,0.000000,0.253687,0.001065,0.000000,0.000000,0.025603,277.072984,0.082375,0.005825,0.064505,0.000000,0.059419,0.009257,2.260954,1.282709,26.475735,52.497414,50.602265,12.865344,61.356703,14.211247,3.183749,19.272370,2.677980,19.572317,281.993722,4.445943,13.854974,1.592477,9.030058,1.218639,548.375201,0.399287,1.191512,2.409710,0.880463,0.429174,0.536705,63.427197,NaN,NaN,NaN
5,NaN,5-3,NaN,4.042,0.202,0.051,0.014,40.418,54.532,0.003,0.026918,0.341804,98.014,4.159746,1.804047,5.829671,0.563635,3.486123,389736.547789,1.959337,0.120013,0.033693,1.540896,6.459337,1.264953,1.110391,2.435068,3.383274,0.152614,1145.616275,3.845851,0.042242,0.530115,0.014581,0.000000,0.000000,0.284840,0.000000,0.000000,334.126707,0.069449,0.004260,0.052186,0.000000,0.000000,0.000000,1.748915,1.601665,24

In [5]:

# 5. Examine all column names

print("Total number of columns:", len(raw_data.columns))

for number, column in enumerate(raw_data.columns, start=1):
    print(number, ":", column)


Total number of columns: 75
1 : Unnamed: 0
2 : Points
3 : Mineral phase 
4 : F
5 : Na2O 
6 : MgO
7 : FeO
8 : P2O5
9 : CaO
10 : MnO
11 : Al2O3
12 : SiO2
13 : Total
14 : Li
15 : Be
16 : B
17 : Sc
18 : V
19 : Ca
20 : Cr
21 : Co
22 : Ni
23 : Cu
24 : Zn
25 : Ga
26 : Ge
27 : As
28 : Se
29 : Rb
30 : Sr
31 : Zr
32 : Nb
33 : Mo
34 : Ag
35 : Cd
36 : In
37 : Sn
38 : Sb
39 : Cs
40 : Ba
41 : Hf
42 : Ta
43 : W
44 : Au
45 : Tl
46 : Bi
47 : Pb
48 : Th
49 : U
50 : La
51 : Ce
52 : Pr
53 : Nd
54 : Sm
55 : Eu
56 : Gd
57 : Tb
58 : Dy
59 : Y
60 : Ho
61 : Er
62 : Tm
63 : Yb
64 : Lu
65 : ∑REY
66 : Ce/Ce* 
67 : Pr/Pr* 
68 : Y/Y*
69 : Eu/Eu*
70 : La/Yb
71 : La/Sm
72 : Y/Ho
73 : 87Sr/86Sr Ini
74 : Sr/Ca
75 : Rb/Sr


In [6]:

# 6. Create a working copy for label inspection

data = raw_data.copy()

# Rename the first three columns for clarity

data = data.rename(columns={
    "Unnamed: 0": "Stratigraphic_unit",
    "Points": "Point_ID",
    "Mineral phase ": "Mineral_phase"
})

# Fill down the merged-cell labels in the working copy

data["Stratigraphic_unit"] = data["Stratigraphic_unit"].ffill()

data["Mineral_phase"] = (
    data["Mineral_phase"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .ffill()
)

# Display label distribution

print("Mineral-phase distribution:")

display(
    data["Mineral_phase"]
    .value_counts()
    .to_frame("Number_of_observations")
)

print("\nStratigraphic distribution:")

display(
    data.groupby(
        ["Stratigraphic_unit", "Mineral_phase"],
        observed=True
    ).size().to_frame("Observations")
)


Mineral-phase distribution:


,Number_of_observations
Mineral_phase,
Oolite core,35
Oolite Rim,27
Pellet Rim,21
Pellet Core,20



Stratigraphic distribution:


Observations
Stratigraphic_unit Mineral_phase              
LPL                Oolite Rim                6
                   Oolite core               8
                   Pellet Core               3
                   Pellet Rim                4
MPL                Oolite Rim                7
                   Oolite core              10
                   Pellet Core               8
                   Pellet Rim                8
UPL                Oolite Rim               14
                   Oolite core              17
                   Pellet Core               9
                   Pellet Rim                9

In [7]:

# 7. Dataset structure and data types

print("Dataset dimensions:", data.shape)

print("\nColumn names and data types:")

display(
    pd.DataFrame({
        "Column": data.columns,
        "Data_Type": data.dtypes.astype(str).values
    })
)

print("\nFirst 10 analytical point identifiers:")

display(
    data[[
        "Stratigraphic_unit",
        "Point_ID",
        "Mineral_phase"
    ]].head(10)
)


Dataset dimensions: (103, 75)

Column names and data types:


,Column,Data_Type
0,Stratigraphic_unit,str
1,Point_ID,str
2,Mineral_phase,string
3,F,float64
4,Na2O,float64
5,MgO,float64
6,FeO,float64
7,P2O5,float64
8,CaO,float64
9,MnO,float64



First 10 analytical point identifiers:


,Stratigraphic_unit,Point_ID,Mineral_phase
0,LPL,2-1,Oolite core
1,LPL,2-2,Oolite core
2,LPL,2-3,Oolite core
3,LPL,5-1,Oolite core
4,LPL,5-2,Oolite core
5,LPL,5-3,Oolite core
6,LPL,6-1,Oolite core
7,LPL,6-2,Oolite core
8,LPL,4-1,Oolite Rim
9,LPL,4-2,Oolite Rim


In [8]:

# 8. Missing value analysis

missing_summary = pd.DataFrame({
    "Missing_Count": data.isna().sum(),
    "Missing_Percentage": (
        data.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_summary)



,Missing_Count,Missing_Percentage
87Sr/86Sr Ini,76,73.79
Rb/Sr,76,73.79
Sr/Ca,76,73.79
F,0,0.00
Na2O,0,0.00
MgO,0,0.00
FeO,0,0.00
P2O5,0,0.00
Stratigraphic_unit,0,0.00
Point_ID,0,0.00


In [9]:

# 9. Examine analytical point identifiers

print("Total observations:", len(data))

print(
    "Duplicate point identifiers within stratigraphic units:",
    data.duplicated(
        subset=["Stratigraphic_unit", "Point_ID"]
    ).sum()
)

print("\nObservations by stratigraphic unit:")

display(
    data.groupby("Stratigraphic_unit").size()
)

print("\nObservations by mineral phase:")

display(
    data.groupby("Mineral_phase").size()
)


Total observations: 103
Duplicate point identifiers within stratigraphic units: 41

Observations by stratigraphic unit:


Stratigraphic_unit
LPL    21
MPL    33
UPL    49
dtype: int64


Observations by mineral phase:


Mineral_phase
Oolite Rim     27
Oolite core    35
Pellet Core    20
Pellet Rim     21
dtype: int64


## 5. Frozen Evaluation Protocol

### 5.1 Classification Task

The task is a four-class classification problem using in-situ geochemical measurements of phosphatic components.

The target variable is Mineral_phase, consisting of:
- Oolite core
- Oolite Rim
- Pellet Core
- Pellet Rim

### 5.2 Data Splitting Strategy

Stratified Group 5-Fold Cross-Validation will be used.

- Number of folds: 5
- Random seed: 42
- Grouping variable: Stratigraphic unit combined with Point ID.
- The same folds will be used for all three models.

This grouping strategy is intended to prevent linked analytical points from appearing in both training and testing partitions.

### 5.3 Model Comparison

Three classification models will be evaluated:
1. Logistic Regression
2. Random Forest
3. Support Vector Machine

All models will use the same evaluation folds and performance metrics.

### 5.4 Evaluation Metrics

The evaluation will include:
- Precision-Recall curves
- ROC curves
- Brier score
- Expected Calibration Error (ECE)
- Calibration curves
- Uncertainty intervals
- Classification error analysis

### 5.5 Reproducibility

The random seed, data version, model configurations, evaluation folds and software dependencies will be documented and retained.


In [10]:

# 10. Create the frozen evaluation folds

from sklearn.model_selection import StratifiedGroupKFold

# Create a composite grouping identifier
data["Group_ID"] = (
    data["Stratigraphic_unit"].astype(str)
    + "_"
    + data["Point_ID"].astype(str)
)

# Define target and groups
y = data["Mineral_phase"]
groups = data["Group_ID"]

# Configure cross-validation
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Assign each observation to one test fold
fold_assignment = np.zeros(len(data), dtype=int)

for fold, (train_idx, test_idx) in enumerate(
    cv.split(data, y, groups),
    start=1
):
    fold_assignment[test_idx] = fold

# Store fold assignments
data["Fold"] = fold_assignment

# Verify the assignments
print("Number of observations:", len(data))
print("Number of folds:", data["Fold"].nunique())

display(
    pd.crosstab(
        data["Fold"],
        data["Mineral_phase"]
    )
)


Number of observations: 103
Number of folds: 5


Mineral_phase,Oolite Rim,Oolite core,Pellet Core,Pellet Rim
Fold,,,,
1,5,7,4,4
2,6,7,4,4
3,5,7,4,4
4,4,7,4,5
5,7,7,4,4


In [11]:

# 11. Verify group separation

for fold in range(1, 6):

    train_groups = set(
        data.loc[data["Fold"] != fold, "Group_ID"]
    )

    test_groups = set(
        data.loc[data["Fold"] == fold, "Group_ID"]
    )

    overlap = train_groups.intersection(test_groups)

    print(
        f"Fold {fold}: "
        f"Overlapping groups = {len(overlap)}"
    )

    assert len(overlap) == 0

print("\nAll five folds passed the group-separation check.")


Fold 1: Overlapping groups = 0
Fold 2: Overlapping groups = 0
Fold 3: Overlapping groups = 0
Fold 4: Overlapping groups = 0
Fold 5: Overlapping groups = 0

All five folds passed the group-separation check.


In [12]:

# 12. Save frozen fold assignments

fold_file = Path(
    r"E:\DS_4\results\frozen_fold_assignments.csv"
)

data[[
    "Stratigraphic_unit",
    "Point_ID",
    "Mineral_phase",
    "Group_ID",
    "Fold"
]].to_csv(
    fold_file,
    index=False
)

print("Frozen fold assignments saved successfully.")
print("Location:", fold_file)


Frozen fold assignments saved successfully.
Location: E:\DS_4\results\frozen_fold_assignments.csv


In [13]:

# 13. Standardize column names

data.columns = data.columns.astype(str).str.strip()

# Define metadata columns
metadata_columns = [
    "Stratigraphic_unit",
    "Point_ID",
    "Mineral_phase",
    "Group_ID",
    "Fold"
]

# Display all available variables
print("Available dataset columns:")

for i, col in enumerate(data.columns, start=1):
    print(i, ":", col)


Available dataset columns:
1 : Stratigraphic_unit
2 : Point_ID
3 : Mineral_phase
4 : F
5 : Na2O
6 : MgO
7 : FeO
8 : P2O5
9 : CaO
10 : MnO
11 : Al2O3
12 : SiO2
13 : Total
14 : Li
15 : Be
16 : B
17 : Sc
18 : V
19 : Ca
20 : Cr
21 : Co
22 : Ni
23 : Cu
24 : Zn
25 : Ga
26 : Ge
27 : As
28 : Se
29 : Rb
30 : Sr
31 : Zr
32 : Nb
33 : Mo
34 : Ag
35 : Cd
36 : In
37 : Sn
38 : Sb
39 : Cs
40 : Ba
41 : Hf
42 : Ta
43 : W
44 : Au
45 : Tl
46 : Bi
47 : Pb
48 : Th
49 : U
50 : La
51 : Ce
52 : Pr
53 : Nd
54 : Sm
55 : Eu
56 : Gd
57 : Tb
58 : Dy
59 : Y
60 : Ho
61 : Er
62 : Tm
63 : Yb
64 : Lu
65 : ∑REY
66 : Ce/Ce*
67 : Pr/Pr*
68 : Y/Y*
69 : Eu/Eu*
70 : La/Yb
71 : La/Sm
72 : Y/Ho
73 : 87Sr/86Sr Ini
74 : Sr/Ca
75 : Rb/Sr
76 : Group_ID
77 : Fold


In [14]:

# 14. Missing-value assessment

geochemical_data = data.drop(
    columns=metadata_columns,
    errors="ignore"
).apply(pd.to_numeric, errors="coerce")

missing_report = pd.DataFrame({
    "Missing_Count": geochemical_data.isna().sum(),
    "Missing_Percentage": (
        geochemical_data.isna().mean() * 100
    ).round(2)
})

display(
    missing_report.sort_values(
        "Missing_Percentage",
        ascending=False
    )
)


,Missing_Count,Missing_Percentage
Rb/Sr,76,73.79
87Sr/86Sr Ini,76,73.79
Sr/Ca,76,73.79
F,0,0.00
P2O5,0,0.00
Na2O,0,0.00
MgO,0,0.00
FeO,0,0.00
Al2O3,0,0.00
MnO,0,0.00


In [15]:

# 15. Define candidate geochemical predictors

epma_features = [
    "F", "Na2O", "MgO", "FeO", "P2O5",
    "CaO", "MnO", "Al2O3", "SiO2"
]

# LA-ICP-MS elemental variables
la_icpms_features = [
    col for col in geochemical_data.columns
    if col in [
        "Li", "Be", "B", "Sc", "V", "Ca", "Cr",
        "Co", "Ni", "Cu", "Zn", "Ga", "Ge", "As",
        "Se", "Rb", "Sr", "Zr", "Nb", "Mo", "Ag",
        "Cd", "In", "Sn", "Sb", "Cs", "Ba", "Hf",
        "Ta", "W", "Au", "Tl", "Bi", "Pb", "Th",
        "U", "La", "Ce", "Pr", "Nd", "Sm", "Eu",
        "Gd", "Tb", "Dy", "Y", "Ho", "Er", "Tm",
        "Yb", "Lu"
    ]
]

candidate_features = (
    epma_features + la_icpms_features
)

print("Number of EPMA predictors:", len(epma_features))
print("Number of LA-ICP-MS predictors:", len(la_icpms_features))
print("Total candidate predictors:", len(candidate_features))

display(
    pd.DataFrame({
        "Predictor": candidate_features,
        "Data_Type": (
            ["EPMA"] * len(epma_features)
            + ["LA-ICP-MS"] * len(la_icpms_features)
        )
    })
)


Number of EPMA predictors: 9
Number of LA-ICP-MS predictors: 51
Total candidate predictors: 60


,Predictor,Data_Type
0,F,EPMA
1,Na2O,EPMA
2,MgO,EPMA
3,FeO,EPMA
4,P2O5,EPMA
5,CaO,EPMA
6,MnO,EPMA
7,Al2O3,EPMA
8,SiO2,EPMA
9,Li,LA-ICP-MS


In [16]:

# 16. Examine candidate predictor availability

X_candidate = data[candidate_features].apply(
    pd.to_numeric,
    errors="coerce"
)

y = data["Mineral_phase"]

print("Candidate predictor dimensions:", X_candidate.shape)

print("\nMissing values:", X_candidate.isna().sum().sum())

print("\nTarget distribution:")
display(y.value_counts().to_frame("Observations"))

print("\nPredictor summary statistics:")
display(X_candidate.describe().T)


Candidate predictor dimensions: (103, 60)

Missing values: 0

Target distribution:


,Observations
Mineral_phase,
Oolite core,35
Oolite Rim,27
Pellet Rim,21
Pellet Core,20



Predictor summary statistics:


,count,mean,std,min,25%,50%,75%,max
F,103.0,3.802767,0.372268,2.471000,3.692500,3.816000,3.993000,4.708000
Na2O,103.0,0.225107,0.079634,0.000000,0.186000,0.227000,0.283000,0.413000
MgO,103.0,0.072913,0.026591,0.001000,0.059000,0.073000,0.090000,0.132000
FeO,103.0,0.020786,0.018746,0.000000,0.008000,0.017000,0.028000,0.127000
P2O5,103.0,40.072388,0.591678,38.569000,39.678000,40.136000,40.484000,41.669000
CaO,103.0,54.305670,0.466699,52.559000,54.105000,54.407000,54.588000,55.187000
MnO,103.0,0.008291,0.010303,0.000000,0.000000,0.006000,0.012500,0.058000
Al2O3,103.0,0.065265,0.093480,0.002165,0.017383,0.035102,0.067738,0.679992
SiO2,103.0,0.418753,0.533909,0.057744,0.152875,0.234161,0.487021,3.968171
Li,103.0,3.993219,1.637826,0.805974,2.871973,3.954781,4.923241,7.894605


In [17]:

# 17. Define candidate geochemical predictor variables

# EPMA major-element measurements
epma_features = [
    "F", "Na2O", "MgO", "FeO", "P2O5",
    "CaO", "MnO", "Al2O3", "SiO2"
]

# LA-ICP-MS elemental measurements
la_icpms_features = [
    "Li", "Be", "B", "Sc", "V", "Ca", "Cr",
    "Co", "Ni", "Cu", "Zn", "Ga", "Ge", "As",
    "Se", "Rb", "Sr", "Zr", "Nb", "Mo", "Ag",
    "Cd", "In", "Sn", "Sb", "Cs", "Ba", "Hf",
    "Ta", "W", "Au", "Tl", "Bi", "Pb", "Th",
    "U", "La", "Ce", "Pr", "Nd", "Sm", "Eu",
    "Gd", "Tb", "Dy", "Y", "Ho", "Er", "Tm",
    "Yb", "Lu"
]

candidate_features = epma_features + la_icpms_features

# Verify that all requested columns exist
missing_columns = [
    col for col in candidate_features
    if col not in data.columns
]

print("EPMA variables:", len(epma_features))
print("LA-ICP-MS variables:", len(la_icpms_features))
print("Total candidate predictors:", len(candidate_features))
print("Unavailable columns:", missing_columns)


EPMA variables: 9
LA-ICP-MS variables: 51
Total candidate predictors: 60
Unavailable columns: []


In [18]:

# 18. Examine candidate predictor quality

X_candidate = data[candidate_features].apply(
    pd.to_numeric,
    errors="coerce"
)

feature_quality = pd.DataFrame({
    "Missing_values": X_candidate.isna().sum(),
    "Missing_percentage": (
        X_candidate.isna().mean() * 100
    ).round(2),
    "Unique_values": X_candidate.nunique(),
    "Standard_deviation": X_candidate.std()
})

display(feature_quality)

print("Total candidate features:", X_candidate.shape[1])
print("Total observations:", X_candidate.shape[0])
print("Total missing measurements:", X_candidate.isna().sum().sum())


,Missing_values,Missing_percentage,Unique_values,Standard_deviation
F,0,0.0,98,0.372268
Na2O,0,0.0,87,0.079634
MgO,0,0.0,62,0.026591
FeO,0,0.0,42,0.018746
P2O5,0,0.0,103,0.591678
CaO,0,0.0,98,0.466699
MnO,0,0.0,25,0.010303
Al2O3,0,0.0,103,0.093480
SiO2,0,0.0,99,0.533909
Li,0,0.0,103,1.637826


Total candidate features: 60
Total observations: 103
Total missing measurements: 0


In [19]:

# 19. Examine repeated analytical identifiers

identifier_check = (
    data.groupby(
        ["Stratigraphic_unit", "Point_ID"]
    )
    .agg(
        Number_of_observations=("Mineral_phase", "size"),
        Number_of_phases=("Mineral_phase", "nunique"),
        Mineral_phases=(
            "Mineral_phase",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
    .reset_index()
)

repeated_identifiers = identifier_check[
    identifier_check["Number_of_observations"] > 1
]

print("Total unique identifier groups:", len(identifier_check))
print("Repeated identifier groups:", len(repeated_identifiers))

display(repeated_identifiers)


Total unique identifier groups: 62
Repeated identifier groups: 32


,Stratigraphic_unit,Point_ID,Number_of_observations,Number_of_phases,Mineral_phases
3,LPL,4-1,2,2,"Oolite Rim, Pellet Core"
4,LPL,4-2,2,2,"Oolite Rim, Pellet Core"
5,LPL,5-1,2,2,"Oolite core, Pellet Core"
8,LPL,6-1,2,2,"Oolite core, Pellet Rim"
9,LPL,6-2,2,2,"Oolite core, Pellet Rim"
20,MPL,13-1,2,2,"Oolite Rim, Pellet Core"
21,MPL,13-2,2,2,"Oolite Rim, Pellet Core"
22,MPL,14-1,2,2,"Oolite core, Pellet Core"
23,MPL,14-2,2,2,"Oolite core, Pellet Core"
24,MPL,15-1,2,2,"Oolite core, Pellet Core"


In [20]:

# 20. Check class distribution across five folds

fold_distribution = pd.crosstab(
    data["Fold"],
    data["Mineral_phase"]
)

print("Mineral-phase distribution across folds:")
display(fold_distribution)

print("\nTotal observations in each fold:")
display(data["Fold"].value_counts().sort_index())


Mineral-phase distribution across folds:


Mineral_phase,Oolite Rim,Oolite core,Pellet Core,Pellet Rim
Fold,,,,
1,5,7,4,4
2,6,7,4,4
3,5,7,4,4
4,4,7,4,5
5,7,7,4,4



Total observations in each fold:


Fold
1    20
2    21
3    20
4    20
5    22
Name: count, dtype: int64

In [21]:

# 21. Verify group separation across folds

for fold in sorted(data["Fold"].unique()):

    train_groups = set(
        data.loc[data["Fold"] != fold, "Group_ID"]
    )

    test_groups = set(
        data.loc[data["Fold"] == fold, "Group_ID"]
    )

    overlap = train_groups.intersection(test_groups)

    print(f"Fold {fold}: {len(overlap)} overlapping groups")

    assert len(overlap) == 0

print("\nGroup separation verification completed.")


Fold 1: 0 overlapping groups
Fold 2: 0 overlapping groups
Fold 3: 0 overlapping groups
Fold 4: 0 overlapping groups
Fold 5: 0 overlapping groups

Group separation verification completed.


In [22]:

# 22. Save the frozen fold assignments

fold_file = Path(
    r"E:\DS_4\results\frozen_fold_assignments.csv"
)

data[
    [
        "Stratigraphic_unit",
        "Point_ID",
        "Mineral_phase",
        "Group_ID",
        "Fold"
    ]
].to_csv(fold_file, index=False)

print("Evaluation folds saved:")
print(fold_file)


Evaluation folds saved:
E:\DS_4\results\frozen_fold_assignments.csv


In [23]:
display(pd.crosstab(data["Fold"], data["Mineral_phase"]))

Mineral_phase,Oolite Rim,Oolite core,Pellet Core,Pellet Rim
Fold,,,,
1,5,7,4,4
2,6,7,4,4
3,5,7,4,4
4,4,7,4,5
5,7,7,4,4


In [24]:
for fold in sorted(data["Fold"].unique()):

    train_groups = set(
        data.loc[data["Fold"] != fold, "Group_ID"]
    )

    test_groups = set(
        data.loc[data["Fold"] == fold, "Group_ID"]
    )

    overlap = train_groups.intersection(test_groups)

    print(f"Fold {fold}: {len(overlap)} overlapping groups")

    assert len(overlap) == 0

print("All folds passed group-separation checks.")

Fold 1: 0 overlapping groups
Fold 2: 0 overlapping groups
Fold 3: 0 overlapping groups
Fold 4: 0 overlapping groups
Fold 5: 0 overlapping groups
All folds passed group-separation checks.
